# Data Cleaning

## Import Required Libraries

In [25]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

## 1. clean_orders() - Fix date formats, handle NULL customer_ids

In [26]:
df = pd.read_csv("orders.csv")
df.head()

,order_id,customer_id,order_date,status,region_code
0,1,409.0,2025-04-12 23:32:22,CANCELLED,EAST
1,2,364.0,2025-10-05 09:19:09,DELIVERED,EAST
2,3,420.0,2024-09-13 13:01:00,SHIPPED,SOUTH
3,4,320.0,2026-04-27 13:29:14,CANCELLED,EAST
4,5,307.0,2026-03-18 14:54:07,CANCELLED,NORTH


In [27]:
print(f"Number of rows are: {df.shape[0]}")
print(f"Number of columns are: {df.shape[1]}")

Number of rows are: 500
Number of columns are: 5


In [28]:
duplicate_values = df[df.duplicated].sum()
duplicate_values

order_id         0
customer_id    0.0
order_date       0
status           0
region_code      0
dtype: object

In [29]:
df.isnull().sum()

order_id        0
customer_id    23
order_date      0
status          0
region_code     0
dtype: int64

In [30]:
def clean_orders(df):
    df["customer_id"] = df["customer_id"].fillna("UNKNOWN")
    df["order_date"] = pd.to_datetime(df["order_date"],format="mixed").dt.strftime("%Y-%m-%d %H:%M:%S")
    return df

df = clean_orders(df)

In [31]:
print("After handling null values: ")
df.isnull().sum()

After handling null values: 


order_id       0
customer_id    0
order_date     0
status         0
region_code    0
dtype: int64

In [32]:
df.to_csv("cleaned_orders.csv",index=False)

## 2. clean_products() - Normalize product names (trim spaces, title case)

In [33]:
df2= pd.read_csv("products.csv")
df2.head()

,product_id,product_name,category,subcategory,cost_price
0,1,Chair,Home,Furniture,4544
1,2,Sony,Electronics,Camera,3852
2,3,Jeans,Clothing,Men,4554
3,4,Shirt,Clothing,Kids,2661
4,5,Harry Potter,Books,Fiction,3892


In [34]:
print(f"Number of rows are: {df2.shape[0]}")
print(f"Number of columns are: {df2.shape[1]}")

Number of rows are: 500
Number of columns are: 5


In [35]:
duplicate_values = df2[df2.duplicated].sum()
duplicate_values

product_id      0
product_name    0
category        0
subcategory     0
cost_price      0
dtype: object

In [36]:
df.isnull().sum()

order_id       0
customer_id    0
order_date     0
status         0
region_code    0
dtype: int64

In [37]:
def clean_products(df):
    df["product_name"] = df["product_name"].str.strip()
    df["product_name"] = df["product_name"].str.title()
    return df

df2 = clean_products(df2)

In [38]:
df2.to_csv("cleaned_products.csv",index=False)

## 3. validate_emails() - Return list of customer_ids with invalid emails

In [17]:
df3= pd.read_csv("customers.csv")
df3.head()

,customer_id,customer_name,email,registration_date,customer_type
0,1,Ronnie Sullivan,johnsonjeffrey@example.net,2024-06-23,PREMIUM
1,2,Matthew Pena,xbrooks@example.net,2023-09-01,REGULAR
2,3,Andrew Cook,wheelerthomas@example.com,2023-12-14,REGULAR
3,4,Kelli Perez,erica48@example.net,2023-12-18,REGULAR
4,5,Robert Marquez,joel26@example.org,2023-07-17,REGULAR


In [18]:
print(f"Number of rows are: {df3.shape[0]}")
print(f"Number of columns are: {df3.shape[1]}")

Number of rows are: 500
Number of columns are: 5


In [19]:
duplicate_values = df3[df3.duplicated].sum()
duplicate_values

customer_id          0
customer_name        0
email                0
registration_date    0
customer_type        0
dtype: object

In [20]:
df3.isnull().sum()

customer_id          0
customer_name        0
email                0
registration_date    0
customer_type        0
dtype: int64

In [21]:
def validate_emails(df):
    invalid_email = df[~df["email"].str.contains("@") |
                       ~df["email"].str.contains(".com",regex=False)]
    return invalid_email[["customer_id","email"]]


In [22]:
validate_emails(df3)

,customer_id,email
0,1,johnsonjeffrey@example.net
1,2,xbrooks@example.net
3,4,erica48@example.net
4,5,joel26@example.org
5,6,aguirrelori@example.org
...,...,...
493,494,jonathansanders@example.org
494,495,ujackson@example.org
496,497,randy73@example.net
498,499,stephanielynch@example.net


In [39]:
df3.to_csv("cleaned_customers.csv")

## 4. check_referential_integrity() - Find order_items that reference non-existent orders

In [40]:
order_items = pd.read_csv("order_itmes.csv")
orders=pd.read_csv("orders.csv")

In [41]:
def check_referential_integrity(order_items,orders):
    invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
    return invalid_orders

df4=check_referential_integrity(order_items,orders)
df4

,item_id,order_id,product_id,quantity,unit_price,discount_percent


In [44]:
order_items.to_csv("cleaned_order_items")